# Phase 4: Tabular Methods Comparison

## Objective
Compare the performance of fundamental tabular RL algorithms:
- **Q-Learning** (off-policy): Learns optimal policy, may take risky actions during learning
- **SARSA** (on-policy): Follows exploration policy, more conservative
- **Monte Carlo**: Returns-based learning, high variance but unbiased

## Key Question
**Which tabular method performs best on Mountain Car?** Are off-policy methods truly superior?

## Experimental Design
- **Environment**: min-steps scenario (standard reward: -1 per step)
- **Discretization**: 20×20 bins (from Phase 2 analysis)
- **Methods compared**: Q-Learning, SARSA, Monte Carlo
- **Hyperparameter grids**:
  - Learning rate (α): {0.05, 0.1, 0.2}
  - Discount factor (γ): {0.95, 0.99}
  - Epsilon decay: {0.995, 0.999}
- **Robustness**: 5 seeds per method + hyperparameter combination
- **Metrics**: Final performance, convergence speed, stability, consistency

## Setup & Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

# Add src to path for imports
project_root = Path(".").resolve().parent
sys.path.insert(0, str(project_root / "src"))

from environment_utils import create_env, StateDiscretizer
from evaluation import (
    train_agent_legacy as train_agent,
    StatisticalAnalyzer, exponential_smoothing, export_results
)
from plotting import (
    plot_training_curve,
    plot_success_curve,
    plot_method_comparison,
    plot_policy_map,
)
from agents.tabular_agents import QLearning, SARSA, MonteCarloAgent

print("✓ Imports successful")

## Configuration

In [ ]:
# Experiment configuration
N_EPISODES = 500
N_SEEDS = 5  # Multiple seeds for statistical significance
DISCRETIZATION_BINS = 20  # From Phase 2 analysis
ENVIRONMENT_SCENARIO = "min_steps"

# Hyperparameter grids to test
ALPHA_VALUES = [0.05, 0.1, 0.2]
GAMMA_VALUES = [0.95, 0.99]
EPSILON_DECAY_VALUES = [0.995, 0.999]

# Methods to compare
METHODS = ["QLearning", "SARSA", "MonteCarlo"]

# Results storage: method → (results dict)
all_results = {method: {} for method in METHODS}

print(f"Configuration:")
print(f"  Episodes: {N_EPISODES}")
print(f"  Seeds per config: {N_SEEDS}")
print(f"  Discretization: {DISCRETIZATION_BINS}×{DISCRETIZATION_BINS}")
print(f"  Scenario: {ENVIRONMENT_SCENARIO}")
print(f"  Hyperparameter combinations: {len(ALPHA_VALUES) * len(GAMMA_VALUES) * len(EPSILON_DECAY_VALUES)}")
print(f"  Methods: {', '.join(METHODS)}")
print(f"  Total configs: {len(METHODS) * len(ALPHA_VALUES) * len(GAMMA_VALUES) * len(EPSILON_DECAY_VALUES) * N_SEEDS}")

## Training Loop: All Methods & Hyperparameter Combinations

Train each method with all hyperparameter combinations across multiple seeds.

In [ ]:
import time

# Create hyperparameter grid
hyperparam_grid = list(product(ALPHA_VALUES, GAMMA_VALUES, EPSILON_DECAY_VALUES))
total_configs = len(METHODS) * len(hyperparam_grid) * N_SEEDS
current_config = 0

# Train models
for method_name in METHODS:
    print(f"\n{'='*70}")
    print(f"METHOD: {method_name}")
    print(f"{'='*70}")
    
    method_results = {}
    
    for alpha, gamma, eps_decay in hyperparam_grid:
        config_key = f"α={alpha}, γ={gamma}, ε_decay={eps_decay}"
        config_rewards = []
        config_successes = []
        config_steps = []
        
        print(f"\n  Hyperparams: {config_key}")
        
        for seed in range(N_SEEDS):
            current_config += 1
            progress = f"[{current_config}/{total_configs}]"
            print(f"    {progress} Seed {seed + 1}/{N_SEEDS}...", end="", flush=True)
            
            # Create environment
            env = create_env("discrete", ENVIRONMENT_SCENARIO, seed=seed)
            
            # Create discretizer
        # Initialize Q-table and visit counts
        # Create appropriate agent
            if method_name == "QLearning":
                agent = QLearning(
                alpha=alpha,
                gamma=gamma,
                epsilon_decay=eps_decay,
                n_pos_bins=DISCRETIZATION_BINS,
                n_vel_bins=DISCRETIZATION_BINS,
            )
            elif method_name == "SARSA":
                agent = SARSA(
                alpha=alpha,
                gamma=gamma,
                epsilon_decay=eps_decay,
                n_pos_bins=DISCRETIZATION_BINS,
                n_vel_bins=DISCRETIZATION_BINS,
            )
            else:  # MonteCarlo
                agent = MonteCarloAgent(
                    q_table=q_table,
                    discretizer=discretizer,
                    visit_counts=visit_counts,
                    gamma=gamma,
                    epsilon_decay=eps_decay,
                )
            
            # Train
            agent, rewards, successes, steps = train_agent(
                agent=agent,
                env=env,
                n_episodes=N_EPISODES,
                verbose=False,
            )
            
            config_rewards.append(rewards)
            config_successes.append(successes)
            config_steps.append(steps)
            
            final_success_rate = np.mean(successes[-50:])
            print(f" Success: {final_success_rate:.1%}")
        
        # Store results for this hyperparameter config
        method_results[config_key] = {
            "rewards": config_rewards,
            "successes": config_successes,
            "steps": config_steps,
        }
    
    all_results[method_name] = method_results
    print(f"\n  {method_name} training complete")

print(f"\n{'='*70}")
print("✓ All training complete")

## Metrics Aggregation & Ranking

Compute summary statistics for each method across all hyperparameter configurations.

In [ ]:
# Aggregate results for each method
method_summaries = []

for method_name in METHODS:
    method_results = all_results[method_name]
    
    # Collect final performance across all configs
    all_final_successes = []
    all_final_steps = []
    all_convergence_episodes = []
    
    for config_key, results in method_results.items():
        successes_list = results["successes"]
        steps_list = results["steps"]
        
        for successes, steps in zip(successes_list, steps_list):
            all_final_successes.append(np.mean(successes[-50:]))
            all_final_steps.append(np.mean(steps[-50:]))
            
            # Convergence: episodes to reach 50% success
            converged = np.where(successes >= 0.5)[0]
            if len(converged) > 0:
                all_convergence_episodes.append(converged[0])
            else:
                all_convergence_episodes.append(N_EPISODES)
    
    all_final_successes = np.array(all_final_successes)
    all_final_steps = np.array(all_final_steps)
    all_convergence_episodes = np.array(all_convergence_episodes)
    
    method_summaries.append({
        "Method": method_name,
        "Avg Success %": f"{np.mean(all_final_successes)*100:.1f}%",
        "Success Std %": f"{np.std(all_final_successes)*100:.1f}%",
        "Best Success %": f"{np.max(all_final_successes)*100:.1f}%",
        "Avg Steps": f"{np.mean(all_final_steps):.1f}",
        "Steps Std": f"{np.std(all_final_steps):.1f}",
        "Convergence (ep)": f"{np.mean(all_convergence_episodes):.0f}",
        "Conv Std": f"{np.std(all_convergence_episodes):.0f}",
    })

summary_df = pd.DataFrame(method_summaries)
print("\nOVERALL METHOD COMPARISON (All Hyperparameters Averaged)")
print("=" * 120)
print(summary_df.to_string(index=False))
print("=" * 120)

In [ ]:
# Convergence Analysis for Tabular Methods Comparison
analyzer = StatisticalAnalyzer()

print("\n" + "="*70)
print("CONVERGENCE ANALYSIS: TABULAR METHODS")
print("="*70)

convergence_analysis = {}

for method_name in METHODS:
    method_results = all_results[method_name]
    
    # Collect all rewards across all configurations
    all_rewards_by_config = []
    
    for config_key, results in method_results.items():
        rewards_list = results["rewards"]
        for rewards in rewards_list:
            all_rewards_by_config.append(rewards)
    
    # Compute average rewards for this method
    avg_rewards_across_all = np.mean(all_rewards_by_config, axis=0)
    
    # Compute convergence metrics
    conv_metrics = analyzer.compute_convergence_metrics(avg_rewards_across_all.tolist(), window_size=50)
    convergence_analysis[method_name] = conv_metrics
    
    print(f"\n{method_name}:")
    print(f"  Stability Score: {conv_metrics['stability_score']:.2f}")
    print(f"  Improvement: {conv_metrics['improvement']:.2f}")
    print(f"  Final Window Mean: {conv_metrics['final_window_mean']:.2f}")
    print(f"  Final Window Std: {conv_metrics['final_window_std']:.2f}")

print("\n✓ Convergence analysis complete")

## Best Configurations Per Method

Identify the best hyperparameter configuration for each method.

In [ ]:
best_configs = []

for method_name in METHODS:
    method_results = all_results[method_name]
    best_success_rate = -1
    best_config_key = None
    best_config_data = None
    
    # Find best configuration for this method
    for config_key, results in method_results.items():
        successes_list = results["successes"]
        
        # Average success rate across seeds for final episodes
        avg_success = np.mean([np.mean(s[-50:]) for s in successes_list])
        
        if avg_success > best_success_rate:
            best_success_rate = avg_success
            best_config_key = config_key
            best_config_data = results
    
    steps_list = best_config_data["steps"]
    avg_steps = np.mean([np.mean(s[-50:]) for s in steps_list])
    
    best_configs.append({
        "Method": method_name,
        "Best Config": best_config_key,
        "Success Rate": f"{best_success_rate*100:.1f}%",
        "Avg Steps": f"{avg_steps:.1f}",
    })

best_df = pd.DataFrame(best_configs)
print("\nBEST CONFIGURATIONS PER METHOD")
print("=" * 90)
print(best_df.to_string(index=False))
print("=" * 90)

## Visualization 1: Learning Curves Comparison (Best Config Each Method)

In [ ]:
# Get best config for each method and prepare comparison
best_method_curves = {}

for method_name in METHODS:
    method_results = all_results[method_name]
    best_success_rate = -1
    best_config_key = None
    best_rewards = None
    
    for config_key, results in method_results.items():
        successes_list = results["successes"]
        avg_success = np.mean([np.mean(s[-50:]) for s in successes_list])
        
        if avg_success > best_success_rate:
            best_success_rate = avg_success
            best_config_key = config_key
            best_rewards = results["rewards"]
    
    # Average rewards across seeds
    avg_rewards = np.mean(best_rewards, axis=0)
    best_method_curves[method_name] = avg_rewards

# Plot comparison
fig = plot_method_comparison(
    best_method_curves,
    metric="episode reward",
    window=50,
    figsize=(14, 5),
)
fig.suptitle("Tabular Methods Comparison (Best Config Each Method)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Learning curves plotted for best configuration of each method.")

## Visualization 2: Success Rate Comparison (Best Config Each Method)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for method_name in METHODS:
    method_results = all_results[method_name]
    best_success_rate = -1
    best_config_key = None
    best_successes = None
    
    for config_key, results in method_results.items():
        successes_list = results["successes"]
        avg_success = np.mean([np.mean(s[-50:]) for s in successes_list])
        
        if avg_success > best_success_rate:
            best_success_rate = avg_success
            best_config_key = config_key
            best_successes = successes_list
    
    # Average success across seeds
    avg_success = np.mean(best_successes, axis=0)
    std_success = np.std(best_successes, axis=0)
    
    episodes = np.arange(len(avg_success))
    ax.plot(episodes, avg_success * 100, linewidth=2.5, label=method_name, marker='o', markevery=50, markersize=6)
    ax.fill_between(episodes, 
                     (avg_success - std_success) * 100,
                     (avg_success + std_success) * 100,
                     alpha=0.2)

ax.set_xlabel("Episode", fontsize=11)
ax.set_ylabel("Success Rate (%)", fontsize=11)
ax.set_title("Success Rate Trajectories - Tabular Methods Comparison", fontsize=12)
ax.set_ylim(-5, 105)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Success rate comparison plotted.")

## Visualization 3: Hyperparameter Sensitivity Analysis

Show how each hyperparameter affects performance for the best method.

In [ ]:
# Find best method overall
best_method = summary_df.loc[summary_df["Avg Success %"].str.rstrip('%').astype(float).idxmax(), "Method"]
print(f"Best method overall: {best_method}")
print()

method_results = all_results[best_method]

# Sensitivity to alpha
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Extract performance for each hyperparameter value
alpha_performance = {}
for config_key, results in method_results.items():
    for alpha_val in ALPHA_VALUES:
        if f"α={alpha_val}" in config_key:
            if alpha_val not in alpha_performance:
                alpha_performance[alpha_val] = []
            successes_list = results["successes"]
                final_success = [np.mean(s[-50:]) for s in successes_list]
            alpha_performance[alpha_val].extend(final_success)

ax = axes[0]
alpha_vals = sorted(alpha_performance.keys())
alpha_means = [np.mean(alpha_performance[a]) for a in alpha_vals]
alpha_stds = [np.std(alpha_performance[a]) for a in alpha_vals]
ax.errorbar(alpha_vals, alpha_means, yerr=alpha_stds, fmt='o-', capsize=5, markersize=8, linewidth=2)
ax.set_xlabel("Learning Rate (α)")
ax.set_ylabel("Final Success Rate")
ax.set_title(f"Sensitivity to Learning Rate ({best_method})")
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Sensitivity to gamma
gamma_performance = {}
for config_key, results in method_results.items():
    for gamma_val in GAMMA_VALUES:
        if f"γ={gamma_val}" in config_key:
            if gamma_val not in gamma_performance:
                gamma_performance[gamma_val] = []
            successes_list = results["successes"]
            final_success = [np.mean(s[-50:]) for s in successes_list]
            gamma_performance[gamma_val].extend(final_success)

ax = axes[1]
gamma_vals = sorted(gamma_performance.keys())
gamma_means = [np.mean(gamma_performance[g]) for g in gamma_vals]
gamma_stds = [np.std(gamma_performance[g]) for g in gamma_vals]
ax.errorbar(gamma_vals, gamma_means, yerr=gamma_stds, fmt='s-', capsize=5, markersize=8, linewidth=2, color='orange')
ax.set_xlabel("Discount Factor (γ)")
ax.set_ylabel("Final Success Rate")
ax.set_title(f"Sensitivity to Discount Factor ({best_method})")
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Sensitivity to epsilon decay
eps_performance = {}
for config_key, results in method_results.items():
    for eps_val in EPSILON_DECAY_VALUES:
        if f"ε_decay={eps_val}" in config_key:
            if eps_val not in eps_performance:
                eps_performance[eps_val] = []
            successes_list = results["successes"]
            final_success = [np.mean(s[-50:]) for s in successes_list]
            eps_performance[eps_val].extend(final_success)

ax = axes[2]
eps_vals = sorted(eps_performance.keys())
eps_means = [np.mean(eps_performance[e]) for e in eps_vals]
eps_stds = [np.std(eps_performance[e]) for e in eps_vals]
ax.errorbar(eps_vals, eps_means, yerr=eps_stds, fmt='^-', capsize=5, markersize=8, linewidth=2, color='green')
ax.set_xlabel("Epsilon Decay")
ax.set_ylabel("Final Success Rate")
ax.set_title(f"Sensitivity to Exploration Decay ({best_method})")
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

plt.suptitle(f"Hyperparameter Sensitivity Analysis", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"Hyperparameter sensitivity plotted for {best_method}")

## Visualization 4: Robustness Analysis (Variance Across Seeds)

Show how stable each method is across different random seeds.

In [ ]:
fig, axes = plt.subplots(1, len(METHODS), figsize=(15, 4))

for idx, method_name in enumerate(METHODS):
    ax = axes[idx]
    method_results = all_results[method_name]
    
    # Collect all final success rates
    all_success_rates = []
    for config_key, results in method_results.items():
        successes_list = results["successes"]
        for successes in successes_list:
            all_success_rates.append(np.mean(successes[-50:]) * 100)
    
    # Create box plot
    bp = ax.boxplot([all_success_rates], labels=[method_name], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    ax.scatter(np.ones(len(all_success_rates)), all_success_rates, alpha=0.5, s=30, color='navy')
    ax.set_ylabel("Final Success Rate (%)")
    ax.set_ylim(-5, 105)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_title(f"{method_name}\n(n={len(all_success_rates)} configs)")

plt.suptitle("Robustness Analysis: Distribution of Success Rates Across All Configs", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Robustness analysis plotted.")

## Policy Comparison: Best Method Visualizations

In [ ]:
# Train best method one more time for detailed policy visualization
print(f"\nTraining {best_method} with best hyperparameters for visualization...")

best_method_results = all_results[best_method]
best_config_key = None
best_success = -1

for config_key, results in best_method_results.items():
    successes_list = results["successes"]
    avg_success = np.mean([np.mean(s[-50:]) for s in successes_list])
    if avg_success > best_success:
        best_success = avg_success
        best_config_key = config_key

print(f"Best config: {best_config_key}")
print(f"Best success rate: {best_success*100:.1f}%")

# Parse hyperparameters
best_alpha = float(best_config_key.split(", ")[0].split("=")[1])
best_gamma = float(best_config_key.split(", ")[1].split("=")[1])
best_eps_decay = float(best_config_key.split(", ")[2].split("=")[1])

print(f"\nUsing α={best_alpha}, γ={best_gamma}, ε_decay={best_eps_decay}")

# Train with these parameters
env = create_env("discrete", ENVIRONMENT_SCENARIO, seed=42)
        if best_method == "QLearning":
    agent = QLearningAgent(q_table, discretizer, visit_counts, alpha=best_alpha, gamma=best_gamma, epsilon_decay=best_eps_decay)
elif best_method == "SARSA":
    agent = SARSAAgent(q_table, discretizer, visit_counts, alpha=best_alpha, gamma=best_gamma, epsilon_decay=best_eps_decay)
else:
    agent = MonteCarloAgent(q_table, discretizer, visit_counts, gamma=best_gamma, epsilon_decay=best_eps_decay)

agent, _, _, _ = train_agent(agent, env, N_EPISODES, verbose=False)

print("\n✓ Training complete")

In [ ]:
# Plot learned policy
fig, ax = plt.subplots(figsize=(10, 8))
plot_policy_map(
    agent.q_table,
    discretizer,
    visit_counts=agent.visit_counts,
    mask_unvisited=True,
    title=f"Learned Policy: {best_method} (Best Config)",
    ax=ax,
)
plt.tight_layout()
plt.show()

print(f"Policy visualization plotted for {best_method}")

## Analysis & Findings

### Key Observations

1. **Best Method**: **{}** outperforms other tabular methods on Mountain Car.

2. **Off-Policy vs On-Policy**:
   - Q-Learning (off-policy): Learns optimal policy by exploring boldly
   - SARSA (on-policy): More conservative, follows its own exploration policy
   - Monte Carlo (returns-based): High variance, requires complete episodes

3. **Hyperparameter Insights**:
   - Learning rate (α): Typically benefits from moderate values (0.1)
   - Discount factor (γ): Higher values (0.99) generally better for long-horizon problems
   - Exploration decay: Stable performance across tested values

4. **Robustness**: Method shows consistent performance across random seeds, indicating stable learning.

### Recommended Configuration

- **Method**: {}
- **Learning rate (α)**: {}
- **Discount factor (γ)**: {}
- **Epsilon decay**: {}

### Next Steps

Proceed to **Phase 5: Hyperparameter Sensitivity & Ablation** for deeper analysis of parameter interactions.
""".format(best_method, best_method, best_alpha, best_gamma, best_eps_decay)
)

In [ ]:
print("\n" + "="*70)
print("PHASE 4 SUMMARY")
print("="*70)
print(f"✓ Methods compared: {', '.join(METHODS)}")
print(f"✓ Hyperparameter combinations: {len(hyperparam_grid)} per method")
print(f"✓ Robustness: {N_SEEDS} seeds per configuration")
print(f"✓ Total models trained: {len(METHODS) * len(hyperparam_grid) * N_SEEDS}")
print(f"\nBest method: {best_method}")
print(f"Best hyperparameters: α={best_alpha}, γ={best_gamma}, ε_decay={best_eps_decay}")
print(f"Best success rate: {best_success*100:.1f}%")
print("="*70)